# Blindspots — proof of concept

A person says one thing. We show them the tetrad around it, with the corner they are
not looking at — **A+**, the constructive side of the opposing view — in front.

This notebook is the pre-MVP: no app, no server, no UI. It exists to answer two
questions with real output before anyone builds a backend:

1. Is the A+ the framework produces worth showing to a person?
2. Which of the two ways of producing it should the app run?

**Where the probe left it (2026-09-29, Sonnet 5, 20 utterances, one rater —
`tests/probe_blindspot_paths.py`, per-item JSON kept):** Path A drew 20/20 at a median
of 8 s with antitheses that are genuine opposing positions; Path B drew 19/20 at a median
of 49 s, 17 of 19 flagged by the coherence check, with antitheses that were mostly strawman
extremes. That last finding was traced to HS-ranked antithesis selection and fixed the same
day (`docs/dev-notes/antithesis-selection.md`: caricatures 19/19 → 0/20, coherence passes
2/19 → 7/20, median 59 s, a third of antitheses still mirror stances). So the pre-MVP runs
**Path A**. Path B stays in the notebook because it is the
one with scores and memory — the "your recurring blindspots" screen needs it — and because
its antithesis quality is a framework finding to fix, not a reason to forget it.

**Path A** asks the tool-less `Consultant` for the view — one structured turn, no
database. **Path B** runs the checked pipeline `anchor` runs (`AnchorTheses` →
`AnalysisPipeline` → `ExpandPolarity`) inside a scratch Case — taxonomy-anchored
antithesis, HS gate, validation, and a graph that remembers. Same `PerspectiveView`
out of both, so one renderer serves both. See `docs/surfaces.md` ("Drawing it") and
`docs/dev-notes/views.md` for why the shape is what it is.

Needs a `.env` with `DIALEXITY_DEFAULT_MODEL` (and provider credentials). Path B also
needs the graph database: `docker compose -f docker-compose.test.yml up -d`.


In [ ]:
from dialectical_framework.dialectical_reasoning import DialecticalReasoning
from dialectical_framework.settings import Settings

# DI setup, once. Reads .env (host obligation 1 — see examples/advisor_chat.py).
container = DialecticalReasoning.setup(Settings.from_env())
print("model:", container.settings().ai_model)

In [ ]:
UTTERANCE = "I should quit my job and start my own company."


## The renderer

Both paths return `graph/views.py::PerspectiveView`. A consumer app never shows
framework terminology, so the view goes through `without_terminology()` — the same
projection the Advisor applies under `hides_terminology` — and the card names the
corners in plain words. `intent` is the reading (the two axes the tetrad was read
along); `complete` says whether all six corners exist.


In [ ]:
from IPython.display import Markdown, display

from dialectical_framework.graph.views import PerspectiveView


def render(view: PerspectiveView, title: str = "") -> None:
    """One card per perspective: A+ first, the two traps after, the poles as context."""
    v = view.without_terminology()
    text = lambda pole: pole.text if pole else "_(not drawn)_"
    lines = [f"### {title}" if title else ""]
    lines += [
        f"**What you might be missing:** {text(v.a_plus)}",
        "",
        f"- Your position: {text(v.t)}",
        f"- What it stands against: {text(v.a)}",
        f"- Your position, developed well: {text(v.t_plus)}",
        "",
        f"- Trap on your side: {text(v.t_minus)}",
        f"- Trap on the other side: {text(v.a_minus)}",
    ]
    if v.intent:
        lines += ["", f"_{v.intent}_"]
    if not v.complete:
        lines += ["", "_(unfinished tetrad — some corners were not drawn)_"]
    display(Markdown("\n".join(lines)))

## Path A — the Consultant's view turn (no graph) — the pre-MVP path

The utterance is the only message in the conversation; `exploration_view(focus=…)` is
one structured, thinking turn that builds the tetrad and returns it. Nothing is
checked and nothing is kept. Typically 5–25 s.


In [ ]:
from dialectical_framework.agents.apps import COUNSELOR_PERSONA
from dialectical_framework.agents.consultant.consultant import Consultant

FOCUS = (
    "the perspective for what I just said: my position, what it stands against, "
    "and above all the constructive side of that opposing view that I am not seeing"
)


async def blindspot_via_consultant(text: str) -> PerspectiveView | None:
    head = Consultant(app_preamble=COUNSELOR_PERSONA, messages=[{"role": "user", "content": text}])
    view = await head.exploration_view(focus=FOCUS)
    return view.perspectives[0] if view.perspectives else None


a = await blindspot_via_consultant(UTTERANCE)
render(a, "Path A — Consultant") if a else print("nothing drawn")

## Path B — the checked pipeline (graph) — scores and memory; not the pre-MVP path

What the Advisor's `anchor` tool runs, called headlessly inside a Case: the
statement is classified (SIMPLE/COMPLEX decides the antithesis path), the antithesis
is found against the systemic taxonomy and scored (HS), the four aspects are
generated with complementarity scores, the tetrad is validated, and everything is
committed to the graph — so a second utterance from the same person lands next to
the first, and `exploration_view()` becomes their map of recurring blindspots.

The host owns the Case (the framework never creates one) and every call runs inside
`scope(sid)`. Typically 15–40 s; the `sid:progress` channel exists to show the person
what is happening meanwhile.


In [ ]:
import json

from dialectical_framework.agents.advisor.tools.anchor import _anchor
from dialectical_framework.graph.nodes.case import Case
from dialectical_framework.graph.nodes.perspective import Perspective
from dialectical_framework.graph.repositories.node_repository import NodeRepository
from dialectical_framework.graph.scope_context import scope
from dialectical_framework.graph.views import exploration_view, perspective_view

# One Case per person. Kept across calls: that is where the memory lives.
case = Case()
case.commit()
print("sid:", case.sid)


async def blindspot_via_pipeline(text: str) -> PerspectiveView | None:
    with scope(case.sid):
        report = json.loads(await _anchor(thesis=text, antithesis=None, context=""))
        hashes = (report.get("artifacts") or {}).get("perspective_hashes") or []
        if not hashes:
            print("nothing anchored:", report.get("summary"))
            return None
        pp = NodeRepository().find_by_hash(hashes[0], node_type=Perspective)
        return perspective_view(pp)


b = await blindspot_via_pipeline(UTTERANCE)
render(b, "Path B — pipeline") if b else None

### What Path B knows that Path A cannot say

The same view, before `without_terminology()`: the antithesis's HS (how well the
opposition sits against the taxonomy apex), the complementarity of each corner, the
tetrad's SP, and the validation verdict. None of this exists on Path A — its numbers
are `None` because nothing scored them, not because they are zero.


In [ ]:
if b:
    print("HS(A)        :", b.a.hs)
    print("Ks per corner:", {n: round(p.ks, 2) for n, p in b.poles.items() if p.ks is not None})
    print("metrics      :", b.metrics)
    print("validation   :", b.validation)

### The map (Path B only)

Every utterance anchored in this Case is a perspective in the graph. After a few, this
is the person's map — the view a "your blindspots" screen would draw.


In [ ]:
with scope(case.sid):
    the_map = exploration_view()
print(len(the_map.perspectives), "perspective(s) in this Case")
for i, p in enumerate(the_map.perspectives, 1):
    render(p, f"#{i}")

## Where this goes

- **Re-run the probe before trusting it.** `tests/probe_blindspot_paths.py` runs 20
  utterances through both paths and writes every item to JSON; the verdict above is one
  sample. `BLINDSPOT_PATHS=a` reruns the cheap side alone.
- **Backend**: a small HTTP service hosting the chosen path — `POST /blindspot {text}` →
  `PerspectiveView.to_dict()`. Host obligations are the documented ones (Case per
  person, `scope(sid)` per request, `principal="human"`, drain at shutdown).
- **Not a tension**: a bare question may draw nothing. The app needs a graceful
  "nothing to draw yet" state; Path A returns an empty view, Path B reports no anchor.
- **Unmeasured**: A+ quality on either path against real users' phrasing. The probe
  is the first sample, not a result.
